# Demo 2: explicit execution state, a judge, and a focused retry

### The same docs desk over a whole shift. Three ways to carry memory. One verifier that names the failed part.

**The setting.** Demo 1 compiled a surface for one request whose facts arrived
clean. Real work does not. Over a shift of `T` events, changes are merged and later
reverted, a number is re-measured, a timestamp is corrected, chat speculates about a
cause, support names a customer. Two deliverables are due partway through: a
customer incident report and the 2.4.0 release notes.

**Three harnesses, one difference.** They share the events, the same small model on
every step, the same procedure and the same writer. They differ only in what they
carry from one step to the next:

| harness | carries between steps | hands the writer |
|---|---|---|
| `transcript` | every event and response, replayed each step | the whole log |
| `window` | the most recent events that fit a fixed character budget | the window |
| `state` | an explicit state **Σ** (a small JSON document with a schema), validated on every step | Σ: current facts only |

**The new part** (sections 11 and 12): a **judge**, a second model of a different
family, checks a deliverable against the compiled rules and names *one failed
part*. The harness then extracts that part only, re-prompts the writer with that
part, its governing rules and the relevant facts, and runs four deterministic
**gates** before accepting the repair.

Definitions:

- **Σ (sigma), the execution state**: a JSON document the harness owns. Each step the
  model proposes a patch; the runtime validates it against a schema before merging.
- **trap**: a planted event that a correct deliverable must not repeat (a reverted
  change, a superseded number).
- **gate**: a check written in code, not judged by a model.

Architecture reference: Badhe, Tiwari & Chung (2026), *SKILL.state*, arXiv:2608.26263;
runtime: [`skill_state`](https://github.com/Nicolepcx/skill_state), an independent
reference implementation.

## What everyone runs vs. what is replay

Every section heading below carries one of two labels.

**Everyone runs** (about ten model calls, under a minute of model time):

1. setup and the Chapter 3 harness (sections 0 to 2);
2. the merge-rule tests on Σ, no model involved (section 3);
3. **one delayed correction, step by step**: the state harness at the step where the
   perf team re-measures the 40 percent as 35. You see the observation, the patch the
   tracker proposes in one live call, Σ before and after (section 5);
4. the writer input built from that Σ, and one live writer call (sections 7 and 8);
5. the judge and **one focused repair** on the planted case (sections 11 and 12).

**Replay** (`RUN_FULL = False` skips it; the outputs you see were saved from an
execution with `RUN_FULL = True`): the full three-harness comparison, about 110 model
calls and roughly 90 seconds for the shifts, plus the plots, the graded deliverables,
the provenance mutation test, the scorecard and the full-regeneration comparison.
Flip the constant in the next cell to run all of it yourself.

In [20]:
RUN_FULL = False   # False: skip the three-harness comparison and read its saved outputs; True: run everything live

## 0. Setup (everyone runs): Chapter 3's artifacts and harness, unchanged

Installs (including `skill_state` from GitHub), the five artifacts from Drive, the
key, and Chapter 3's harness code verbatim.

In [21]:
%pip install -q instructor gdown python-dotenv pyyaml tiktoken

import importlib, shutil, subprocess, sys

SKILL_STATE_REPO = "https://github.com/Nicolepcx/skill_state"
try:
    import skill_state
except ModuleNotFoundError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{SKILL_STATE_REPO}"],
                       capture_output=True, text=True)
    if r.returncode != 0:                     # repo without pyproject.toml: use a clone
        shutil.rmtree("/tmp/skill_state_src", ignore_errors=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", SKILL_STATE_REPO,
                        "/tmp/skill_state_src"], check=True)
        sys.path.insert(0, "/tmp/skill_state_src")
    importlib.invalidate_caches()
    import skill_state
print("skill_state loaded from", skill_state.__file__)

skill_state loaded from /usr/local/lib/python3.13/dist-packages/skill_state/__init__.py


In [22]:
try:
    import gdown
except ModuleNotFoundError:
    %pip install -q gdown
    import gdown

import hashlib, inspect, json, os, re, shutil, textwrap, time
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import yaml

DRIVE_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1hPCzmDY08tKq5o8vcykap9aFdiQ7UA_4?usp=sharing"
)

WORK = Path("ch03_skill_demo").resolve()

# gdown preserves the Google Drive folder name.
SRC = WORK / "artifacts"

shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

kwargs = {
    "url": DRIVE_FOLDER_URL,
    "output": str(WORK),
    "quiet": True,
    "use_cookies": False,
}

if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
    kwargs["remaining_ok"] = True

downloaded = gdown.download_folder(**kwargs)

if not downloaded:
    raise RuntimeError(
        "Nothing was downloaded. Ensure that the Google Drive folder is "
        "shared with 'Anyone with the link'."
    )

files = sorted(SRC.glob("*.md"))

if not files:
    raise RuntimeError(f"No Markdown artifacts found in {SRC}/")

print(f"{len(files)} artifact(s) in {SRC}:")
for p in files:
    print(f"   {p.name:28} {p.stat().st_size:>6} bytes")

5 artifact(s) in /content/ch03_skill_demo/artifacts:
   AGENTS.md                      1023 bytes
   PROJECT.md                      891 bytes
   persona-plainspoken.md          571 bytes
   skill-incident-report.md       1429 bytes
   skill-release-notes.md         3645 bytes


In [23]:
import getpass, urllib.request

import instructor
import openai
from pydantic import BaseModel, Field

from dotenv import load_dotenv, find_dotenv


def _find_key():
    """OPENROUTER_API_KEY from, in order: the environment, a .env next to the notebook,
    the course repo's course_app/.env, a Colab secret, or a prompt. The value is never printed."""
    if os.environ.get("OPENROUTER_API_KEY"):
        return
    load_dotenv(find_dotenv(usecwd=True), override=False)
    for base in [Path.cwd(), *Path.cwd().parents][:4]:
        if (base / "course_app" / ".env").exists():
            load_dotenv(base / "course_app" / ".env", override=False)
            break
    if not os.environ.get("OPENROUTER_API_KEY"):
        try:
            from google.colab import userdata                 # Colab: the key icon in the left sidebar
            os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
        except Exception:                                      # noqa: BLE001
            pass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")


_find_key()

BASE_URL = "https://openrouter.ai/api/v1"
WRITER = "openai/gpt-5.6-luna"          # writes the notes
ROUTER = "openai/gpt-5.4-mini"          # picks the skill; a small model is plenty

USAGE: List[Dict[str, object]] = []      # every call, for the cost section at the end


@lru_cache(maxsize=4)
def client_for(mode):
    return instructor.from_openai(
        openai.OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL),
        mode=mode)


def typed_call(*, model, response_model, messages, temperature=0.0, label="",
               mode=instructor.Mode.TOOLS, max_retries=2):
    """One structured call, with its usage and wall clock recorded."""
    t0 = time.perf_counter()
    out, completion = client_for(mode).chat.completions.create_with_completion(
        model=model, response_model=response_model, max_retries=max_retries,
        temperature=temperature, messages=messages)
    u = getattr(completion, "usage", None)
    USAGE.append({"label": label or model, "model": model,
                  "prompt_tokens": getattr(u, "prompt_tokens", 0),
                  "completion_tokens": getattr(u, "completion_tokens", 0),
                  "seconds": time.perf_counter() - t0})
    return out


JUDGE = "anthropic/claude-haiku-5.5"    # verifies deliverables; a different model family from the writer

print(f"writer: {WRITER}   router: {ROUTER}   judge: {JUDGE}   key loaded: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

writer: openai/gpt-5.6-luna   router: openai/gpt-5.4-mini   judge: anthropic/claude-haiku-5.5   key loaded: True


In [24]:
# --- chapter 3, verbatim -------------------------------------------------------


_FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n?", re.S)


_SECTION = re.compile(r"^<!--\s*section:\s*([\w.\-]+)\s*-->\s*$", re.M)


@dataclass(frozen=True)
class Artifact:
    kind: str                       # identity | project | persona | skill
    name: str
    version: str
    path: Path
    meta: Dict
    sections: Dict[str, str]        # id -> body, in file order
    digest: str                     # content hash of the file as authored

    @property
    def always_loaded(self) -> bool:
        return bool(self.meta.get("always_loaded"))

    @property
    def owns(self) -> List[str]:
        return list(self.meta.get("owns") or [])

    @property
    def axes(self) -> List[str]:
        return list(self.meta.get("axes") or [])


def parse(path: Path) -> Artifact:
    raw = path.read_text(encoding="utf-8")
    m = _FRONTMATTER.match(raw)
    if not m:
        raise ValueError(f"{path.name}: no frontmatter — not an artifact")
    meta = yaml.safe_load(m.group(1)) or {}
    body = raw[m.end():]

    # Split on the section markers. A body outside any marker is a file someone
    # wrote as prose; it would be unaddressable, so it is an error, not a warning.
    marks = list(_SECTION.finditer(body))
    if not marks:
        raise ValueError(f"{path.name}: no <!-- section: ... --> markers")
    if body[:marks[0].start()].strip():
        raise ValueError(f"{path.name}: text before the first section marker")
    sections = {}
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        sections[mark.group(1)] = body[mark.end():end].strip()

    for missing in set(meta.get("owns") or []) - set(sections):
        raise ValueError(f"{path.name}: claims to own '{missing}' but has no such section")

    return Artifact(kind=meta["artifact"], name=meta["name"], version=str(meta["version"]),
                    path=path, meta=meta, sections=sections,
                    digest=hashlib.sha256(raw.encode()).hexdigest()[:12])


class SkillChoice(BaseModel):
    skill: str = Field(description="The name of the skill to load, exactly as listed.")
    why: str = Field(description="One short sentence.")


def catalogue(descriptions: Dict[str, str]) -> str:
    return "\n".join(f"- {name}: {desc.strip()}" for name, desc in descriptions.items())


def overlap(artifacts: Sequence[Artifact]) -> Dict[str, List[str]]:
    """Section ids claimed by more than one artifact."""
    claims: Dict[str, List[str]] = {}
    for a in artifacts:
        for section in a.owns:
            claims.setdefault(section, []).append(a.name)
    return {sid: names for sid, names in claims.items() if len(names) > 1}


def load_for_task(names: Sequence[str]) -> List[Artifact]:
    """Identity + project + the named artifacts, refusing an ambiguous surface."""
    chosen = ALWAYS + [ARTIFACTS[n] for n in names]
    clash = overlap(chosen)
    if clash:
        raise ValueError(
            "ambiguous surface — every section needs exactly one owner:\n" +
            "\n".join(f"   {sid}: claimed by {', '.join(owners)}"
                      for sid, owners in sorted(clash.items())))
    return chosen


_H3 = re.compile(r"^### (.+)$", re.M)


AXIS_LABELS = {"tone": "Tone", "audience": "Audience"}


def _heading_axes(heading: str) -> Dict[str, str]:
    """`Tone: casual, Audience: customer` -> {'tone': 'casual', 'audience': 'customer'}."""
    found = {}
    for part in heading.split(","):
        if ":" not in part:
            continue
        key, _, value = part.partition(":")
        key = key.strip().lower()
        if key in AXIS_LABELS:
            found[key] = value.strip().lower()
    return found


def slice_section(body: str, axes: Dict[str, str]) -> str:
    """Keep the preamble, and only the H3 blocks whose axes all match."""
    marks = list(_H3.finditer(body))
    if not marks:
        return body
    kept = [body[:marks[0].start()].strip()]
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        heading, block = mark.group(1).strip(), body[mark.end():end].strip()
        wanted = _heading_axes(heading)
        # A heading with no axes at all is unconditional prose, so it stays.
        if wanted and any(axes.get(k) != v for k, v in wanted.items()):
            continue
        kept.append(block)
    return "\n\n".join(x for x in kept if x)


def _norm(line: str) -> str:
    return re.sub(r"\s+", " ", line.strip().lower())


#: Load order, and therefore precedence when the same paragraph appears twice.
#: Durable first: a rule the agent carries everywhere outranks a task-local
#: restatement of it, so the surviving copy is the one with the wider scope.
KIND_PRECEDENCE = ("identity", "project", "skill", "persona")


def in_load_order(artifacts: Sequence[Artifact]) -> List[Artifact]:
    return sorted(artifacts, key=lambda a: KIND_PRECEDENCE.index(a.kind))


@dataclass
class Surface:
    text: str
    manifest: Dict[str, object]

    @property
    def words(self) -> int:
        return len(self.text.split())


def compile_surface(*, artifacts: Sequence[Artifact], axes: Dict[str, str],
                    facts: Dict[str, object]) -> Surface:
    blocks, seen, dropped, used = [], {}, [], []
    for a in in_load_order(artifacts):
        for sid, body in a.sections.items():
            text = slice_section(body, axes) if a.axes else body
            # Deduplicate paragraph by paragraph, not section by section: the
            # same sentence written by two teams is the common case, two
            # identical sections is not. The copy that survives is the one from
            # the artifact earliest in load order, and the manifest records who
            # kept it — so "who says this?" has exactly one answer.
            kept = []
            for para in re.split(r"\n\s*\n", text):
                if not para.strip():
                    continue
                key = _norm(para)
                if key in seen:
                    dropped.append({"dropped_from": f"{a.name}:{sid}",
                                    "kept_from": seen[key],
                                    "text": para.strip()[:70]})
                    continue
                seen[key] = f"{a.name}:{sid}"
                kept.append(para.strip())
            if not kept:
                continue
            blocks.append("\n\n".join(kept))
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}\n\n"
                  "These are given. Do not restate them as claims of your own, and "
                  "do not infer any fact that is not listed.")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={
        "axes": dict(axes), "sections": used, "deduplicated": dropped,
        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


def everything_surface(artifacts: Sequence[Artifact], facts: Dict[str, object]) -> Surface:
    """Every artifact, every variant — the way a hand-maintained prompt grows."""
    blocks, used = [], []
    for a in artifacts:
        for sid, body in a.sections.items():
            blocks.append(body)
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={"axes": {}, "sections": used, "deduplicated": [],
                                        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


class ReleaseNotes(BaseModel):
    summary: str = Field(description="One sentence, at most 25 words.")
    highlights: List[str] = Field(description="Three to six bullets, one change each.")


# --- wiring ---------------------------------------------------------------------
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def ntok(t): return len(_enc.encode(t))
except Exception:
    def ntok(t): return max(1, round(len(t) / 4))

ARTIFACTS = {a.name: a for a in (parse(p) for p in files)}
ALWAYS = [a for a in ARTIFACTS.values() if a.always_loaded]
LOADABLE = [a for a in ARTIFACTS.values() if not a.always_loaded]
AS_WRITTEN = {a.name: a.meta["when_to_use"] for a in LOADABLE if a.kind == "skill"}

print(f"{'artifact':22}{'kind':10}{'version':10}digest")
for a in ARTIFACTS.values():
    print(f"{a.name:22}{a.kind:10}{a.version:10}{a.digest}")

artifact              kind      version   digest
acme-docs-agent       identity  2.1.0     f13e99c92930
acme-data-platform    project   4.0.2     6246496911d7
plainspoken           persona   1.2.0     dc78ba66e414
incident-report       skill     1.1.0     a998ee689b8b
release-notes         skill     3.4.0     9a0b8bce844f


## 1. The workload (everyone runs): a shift of events with seven traps

The facts of Demo 1 arrive as a stream. The cell builds a shift of `T`
events: the core facts at fixed positions, filler (CI, syslog, chat) in between, and
two `write` triggers. Seven traps are planted:

| Trap | A correct deliverable |
|---|---|
| PR-418 merged, then reverted | has no dark mode |
| PR-423 merged, then moved to 2.5.0 | has no retries |
| merges into `maintenance/2.3.x` and `main` | includes none of them |
| PR-415 "40 percent", later re-measured as 35 | says 35 |
| incident event at 14:31, later corrected to 14:36 | says 14:36 |
| chat blames the cloud vendor | states no cause and no vendor |
| support names Globex | names no customer |

**The point.** Somebody has to work out what is *still true* at write time. The
question is who, and when.

**Harness computes.**

In [25]:
import random

@dataclass(frozen=True)
class Event:
    text: str
    kind: str                     # fact | trap | write | noise
    expect: Optional[str] = None  # for kind == "write": the skill that should be routed


CORE = [
    (0.02, Event("Merge queue: PR-412 merged into release/2.4.0 — the CSV exporter was rewritten; exports run about twice as fast.", "fact")),
    (0.05, Event("Merge queue: PR-415 merged into release/2.4.0 — peak memory during export reduced by roughly 40 percent.", "fact")),
    (0.08, Event("Merge queue: PR-418 merged into release/2.4.0 — the web console gets a dark mode.", "fact")),
    (0.10, Event("Merge queue: PR-399 merged into maintenance/2.3.x — fixed timezone handling in the scheduler.", "trap")),
    (0.14, Event("Incident channel: INC-88 opened. 14:02 UTC — ingest API returned 503 for all regions.", "fact")),
    (0.18, Event("Incident channel: 14:20 UTC — traffic shifted to the standby ingest cluster.", "fact")),
    (0.22, Event("Chat (#eng): pretty sure the cloud vendor's load balancer caused this, again.", "trap")),
    (0.26, Event("Incident channel: 14:31 UTC — ingest API recovered in eu-west.", "fact")),
    (0.30, Event("Support: Globex says the outage broke their nightly load.", "trap")),
    (0.34, Event("Merge queue: PR-420 merged into release/2.4.0 — the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}.", "fact")),
    (0.38, Event("Incident channel: correction — the eu-west recovery was at 14:36 UTC, not 14:31 UTC.", "fact")),
    (0.42, Event("Incident channel: 15:07 UTC — all regions recovered. INC-88 resolved.", "fact")),
    (0.46, Event("Comms: INC-88 is resolved. Write the customer-facing incident report now.", "write", "incident-report")),
    (0.52, Event("Merge queue: PR-418 reverted from release/2.4.0 — dark mode is not ready.", "fact")),
    (0.58, Event("Merge queue: PR-421 merged into release/2.4.0 — fixed escaping of quotes inside CSV fields.", "fact")),
    (0.64, Event("Perf team: PR-415 re-measured on production-size exports; the peak memory reduction is roughly 35 percent, not 40.", "fact")),
    (0.70, Event("Merge queue: PR-423 merged into release/2.4.0 — the scheduler retries failed jobs up to 3 times.", "fact")),
    (0.78, Event("Release manager: PR-423 moved to the 2.5.0 milestone; it is not part of 2.4.0.", "fact")),
    (0.86, Event("Release manager: 2.4.0 is flagged breaking=true.", "fact")),
    (1.00, Event("Release manager: version 2.4.0 has shipped. Write the release notes for customers now.", "write", "release-notes")),
]

OTHER_BRANCH = [
    "Merge queue: PR-530 merged into main (targets 2.5.0) — the scheduler accepts cron expressions.",
    "Merge queue: PR-402 merged into maintenance/2.3.x — the web console remembers the last-used export format.",
    "Merge queue: PR-533 merged into main (targets 2.5.0) — the ingest API accepts gzip payloads.",
]
CHAT = ["Chat (#eng): who owns the flaky scheduler test?", "Chat (#eng): lunch order closes at noon.",
        "Chat (#eng): reminder, retro is Thursday.", "Chat (#support): ticket volume is normal today.",
        "Chat (#eng): the staging certificate renews tonight."]


def _noise(rng: random.Random, pool: List[str]) -> Event:
    k = rng.choice(["ci", "syslog", "chat"] + (["branch"] if pool else []))
    if k == "branch":
        return Event(pool.pop(0), "trap")
    if k == "ci":
        return Event(f"CI: build #{rng.randint(4000, 9999)} on main passed in "
                     f"{rng.randint(3, 19)}m{rng.randint(0, 59):02d}s.", "noise")
    if k == "syslog":
        return Event(f"[Syslog] ingest-{rng.randint(1, 40)} CPU load: {rng.randint(5, 99)}%, "
                     f"RAM usage: {rng.randint(20, 95)}%", "noise")
    return Event(rng.choice(CHAT), "noise")


def make_shift(T: int, seed: int = 0) -> List[Event]:
    """T events: the CORE facts at fixed fractions, filler everywhere else."""
    assert T >= len(CORE) + 4, f"horizon must be at least {len(CORE) + 4}"
    rng, pool = random.Random(seed), list(OTHER_BRANCH)
    rng.shuffle(pool)
    shift: List[Optional[Event]] = [None] * T
    last = -1
    for frac, ev in CORE:
        pos = max(round(frac * (T - 1)), last + 1)
        shift[pos], last = ev, pos
    assert last == T - 1, "the release trigger must be the last event"
    return [ev if ev is not None else _noise(rng, pool) for ev in shift]


demo = make_shift(40)
print(f"a 40-event shift: {sum(e.kind == 'fact' for e in demo)} facts, "
      f"{sum(e.kind == 'trap' for e in demo)} traps, {sum(e.kind == 'noise' for e in demo)} noise, "
      f"write triggers at t={[i for i, e in enumerate(demo) if e.kind == 'write']}\n")
for i, e in enumerate(demo[:12]):
    print(f"   t={i:<3}{e.kind:7}{e.text[:100]}")
print("   ...")

a 40-event shift: 15 facts, 5 traps, 18 noise, write triggers at t=[18, 39]

   t=0  noise  CI: build #6121 on main passed in 19m31s.
   t=1  fact   Merge queue: PR-412 merged into release/2.4.0 — the CSV exporter was rewritten; exports run about tw
   t=2  fact   Merge queue: PR-415 merged into release/2.4.0 — peak memory during export reduced by roughly 40 perc
   t=3  fact   Merge queue: PR-418 merged into release/2.4.0 — the web console gets a dark mode.
   t=4  trap   Merge queue: PR-399 merged into maintenance/2.3.x — fixed timezone handling in the scheduler.
   t=5  fact   Incident channel: INC-88 opened. 14:02 UTC — ingest API returned 503 for all regions.
   t=6  trap   Merge queue: PR-530 merged into main (targets 2.5.0) — the scheduler accepts cron expressions.
   t=7  fact   Incident channel: 14:20 UTC — traffic shifted to the standby ingest cluster.
   t=8  noise  Chat (#support): ticket volume is normal today.
   t=9  trap   Chat (#eng): pretty sure the cloud vendor's loa

## 2. The procedure for the shift (everyone runs): a skill like any other

Something must run on every event and say `WAIT` or `WRITE`. Its
instructions are a sixth artifact, `docs-desk`, which goes through the same parse,
boundary check and compile as Demo 1. The compiled result is the procedure **P**,
shared by all three harnesses.

**Harness computes.**

In [26]:
DESK_MD = """---
artifact: skill
name: docs-desk
version: 0.1.0
always_loaded: false
when_to_use: >
  Use while watching the merge queue and the incident channel between
  deliverables: keep track of what is in the release and what happened in the
  incident, and decide when a deliverable is due. Not for writing it.
owns: [desk.inputs, desk.actions]
---

<!-- section: desk.inputs -->
## What you are watching

One event arrives per turn: merge queue, incident channel, release manager,
perf team, support, chat, CI or telemetry.

- A change is in the release only if it was merged into the release branch named
  in the harness facts, and has not since been reverted or moved to another
  milestone. Merges into any other branch are not part of this release.
- A later correction supersedes the statement it corrects: a re-measured number,
  a corrected timestamp.
- Incident events count only when the incident channel reports them with a UTC
  time. Opinions in chat about the cause are not events.
- Chat, CI and telemetry otherwise carry nothing you need.

<!-- section: desk.actions -->
## What you do each turn

Exactly one action:

- `WRITE`: only when the event explicitly asks for a deliverable to be written
  now. The harness then chooses the skill and writes it.
- `WAIT`: every other turn.

Keep your reasoning to at most two sentences.
"""

(SRC / "skill-docs-desk.md").write_text(DESK_MD, encoding="utf-8")
DESK = parse(SRC / "skill-docs-desk.md")                     # chapter 3 §1
ARTIFACTS[DESK.name] = DESK
assert not overlap(ALWAYS + [DESK]), "boundary collision"   # chapter 3 §4

HARNESS_FACTS = {"release_branch": "release/2.4.0"}
P = compile_surface(artifacts=ALWAYS + [DESK], axes={}, facts=HARNESS_FACTS)   # chapter 3 §6
WRITER_AXES = {"release-notes": {"tone": "casual", "audience": "customer"},
               "incident-report": {"audience": "customer"}}

print(f"docs-desk {DESK.version} ({DESK.digest}) owns {DESK.owns}")
print(f"P: {ntok(P.text)} tokens, surface digest {P.manifest['surface_digest']}, "
      f"{len(P.manifest['sections'])} sections")

docs-desk 0.1.0 (09740761965a) owns ['desk.inputs', 'desk.actions']
P: 578 tokens, surface digest d4a429f17914, 8 sections


## 3. Σ is an artifact too: schema, merge rule, tests without a model

The state has a schema owned by the runtime. Each step the model proposes a
patch; `merge` applies it (`null` deletes a key); `validate` checks the result
before it is committed. The cell runs the patches this domain needs, then three
patches the runtime refuses, then one it cannot refuse.

**The point.** Validation checks shape, not truth. A well-formed wrong patch still
gets through; that remains the model's job.

**Harness computes.**

In [27]:
from skill_state import LLMResult, Skill, SkillStateRuntime, build_prompt, merge, validate

STATE_SPEC = """## Execution state

Earlier events will not be shown to you again. The state below is your only memory
of them, so record a fact in the same turn it arrives. Anything you do not record
is lost.

- `release.version`: the version on the release branch.
- `release.breaking`: true or false, only once the release manager states it.
- `release.changes`: {"PR-<n>": "<the change, in the input's words, with its current numbers>"}.
  Only changes currently in this release. Set a key to null when the change is
  reverted or moved out of the release. When a change is corrected, overwrite it
  with the current value only; do not keep the superseded one.
- `incident.id`, `incident.status` ("open" or "resolved").
- `incident.timeline`: {"HH:MM": "<what happened>"}, UTC, recorded events only.
  When a time is corrected, set the old key to null and write the new key."""

STATE_SCHEMA = {
    "type": "object", "required": ["release", "incident"], "additionalProperties": False,
    "properties": {
        "release": {"type": "object", "additionalProperties": False, "properties": {
            "version": {"type": ["string", "null"]},
            "breaking": {"type": ["boolean", "null"]},
            "changes": {"type": "object", "additionalProperties": {"type": "string"}}}},
        "incident": {"type": "object", "additionalProperties": False, "properties": {
            "id": {"type": ["string", "null"]},
            "status": {"type": ["string", "null"], "enum": ["open", "resolved", None]},
            "timeline": {"type": "object", "additionalProperties": {"type": "string"}}}},
    },
}
INITIAL_STATE = {"release": {"changes": {}}, "incident": {"timeline": {}}}


def change_ids(state):
    bad = [k for k in state.get("release", {}).get("changes", {}) if not re.fullmatch(r"PR-\d+", k)]
    return [f"change keys must look like PR-<n>: {bad}"] if bad else []


def timeline_keys(state):
    bad = [k for k in state.get("incident", {}).get("timeline", {}) if not re.fullmatch(r"\d{2}:\d{2}", k)]
    return [f"timeline keys must be HH:MM in UTC: {bad}"] if bad else []


STATE_CHECKS = [change_ids, timeline_keys]


def state_errors(state):
    """Exactly what the runtime checks before it commits a patch."""
    errors = validate(state, STATE_SCHEMA)
    return errors or [e for check in STATE_CHECKS for e in check(state)]


TRACKER_SKILL = Skill(name="docs-desk", instructions=P.text + "\n\n" + STATE_SPEC,
                      schema=STATE_SCHEMA, initial_state=INITIAL_STATE,
                      checks=STATE_CHECKS, fallback_action="WAIT")

s = INITIAL_STATE
for label, patch in [
    ("merge PR-418, PR-415", {"release": {"version": "2.4.0", "changes": {
        "PR-418": "the web console gets a dark mode",
        "PR-415": "peak memory during export reduced by roughly 40 percent"}}}),
    ("record 14:31", {"incident": {"id": "INC-88", "status": "open",
                                   "timeline": {"14:31": "ingest API recovered in eu-west"}}}),
    ("revert PR-418", {"release": {"changes": {"PR-418": None}}}),
    ("re-measure PR-415", {"release": {"changes": {"PR-415": "peak memory during export reduced by roughly 35 percent"}}}),
    ("correct 14:31 -> 14:36", {"incident": {"timeline": {"14:31": None, "14:36": "ingest API recovered in eu-west"}}}),
]:
    s = merge(s, patch)
    assert not state_errors(s), state_errors(s)
    print(f"   [ok] {label:24}{json.dumps(s, separators=(',', ':'))[:110]}")
assert "PR-418" not in s["release"]["changes"] and "35" in s["release"]["changes"]["PR-415"]
assert list(s["incident"]["timeline"]) == ["14:36"]

print("\nrefused before commit (Σ untouched, the model retries):")
for label, patch in [("timeline sent as a list", {"incident": {"timeline": ["14:02 down"]}}),
                     ("timestamp not HH:MM", {"incident": {"timeline": {"2:31pm": "x"}}}),
                     ("a key the schema lacks", {"customers_affected": ["Globex"]})]:
    print(f"   [refused] {label:26}{state_errors(merge(s, patch))}")

wrong = {"release": {"changes": {"PR-415": "peak memory during export reduced by roughly 40 percent"}}}
print(f"\nnot caught: re-recording the superseded 40 -> errors={state_errors(merge(s, wrong))}"
      "\n   validation checks shape, not truth; getting the patch right is still the model's job")

   [ok] merge PR-418, PR-415    {"release":{"changes":{"PR-418":"the web console gets a dark mode","PR-415":"peak memory during export reduced
   [ok] record 14:31            {"release":{"changes":{"PR-418":"the web console gets a dark mode","PR-415":"peak memory during export reduced
   [ok] revert PR-418           {"release":{"changes":{"PR-415":"peak memory during export reduced by roughly 40 percent"},"version":"2.4.0"},
   [ok] re-measure PR-415       {"release":{"changes":{"PR-415":"peak memory during export reduced by roughly 35 percent"},"version":"2.4.0"},
   [ok] correct 14:31 -> 14:36  {"release":{"changes":{"PR-415":"peak memory during export reduced by roughly 35 percent"},"version":"2.4.0"},

refused before commit (Σ untouched, the model retries):
   [refused] timeline sent as a list   ['$.incident.timeline: expected object, got list']
   [refused] timestamp not HH:MM       ["timeline keys must be HH:MM in UTC: ['2:31pm']"]
   [refused] a key the schema lacks    ['$.custo

## 4. The three harnesses

`run_state` uses the `skill_state` runtime; `run_transcript` is the ReAct-style
baseline, and with a character budget it becomes the `window` control. At every
`WRITE`, Chapter 3's `route()` picks the skill and the harness **captures** what the
writer would be handed; writing happens later, from those captures, so all three are
compared at identical moments. This cell only defines them.

In [28]:
import concurrent.futures as cf
import copy

TRACKER = ROUTER                     # one small model runs every step


@lru_cache(maxsize=1)
def raw_client():
    return openai.OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL)


def chat_call(*, model, prompt, label, temperature=0.0):
    t0 = time.perf_counter()
    resp = raw_client().chat.completions.create(
        model=model, temperature=temperature, messages=[{"role": "user", "content": prompt}])
    u = getattr(resp, "usage", None)
    pi, po = getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0
    USAGE.append({"label": label, "model": model, "prompt_tokens": pi,
                  "completion_tokens": po, "seconds": time.perf_counter() - t0})
    return resp.choices[0].message.content or "", pi, po


class OpenRouterLLM:
    """Satisfies skill_state.LLM: complete(prompt) -> LLMResult."""
    def __init__(self, model, label):
        self.model, self.label = model, label

    def complete(self, prompt):
        text, pi, po = chat_call(model=self.model, prompt=prompt, label=self.label)
        return LLMResult(text, pi or None, po or None)


def route(task: str, label: str) -> str:                     # chapter 3 §3, at temperature 0
    return typed_call(model=ROUTER, response_model=SkillChoice, label=label, temperature=0.0,
                      messages=[{"role": "system", "content":
                                 "Choose exactly one skill for the task. Answer with its name."},
                                {"role": "user", "content":
                                 f"SKILLS\n{catalogue(AS_WRITTEN)}\n\nTASK\n{task}"}]).skill


def _verb(action: str) -> str:
    words = re.sub(r"[^A-Za-z]+", " ", action).split()
    return words[0].upper() if words else ""


def _digest(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:12]


def capture(harness, T, seed, t, task, log=None, state=None) -> Dict:
    tag = f"{harness}|T={T}|s={seed}"
    try:
        routed = route(task, f"route[{tag}]")
    except Exception as exc:                                  # noqa: BLE001
        routed = f"error: {exc}"[:120]
    return {"harness": harness, "T": T, "seed": seed, "t": t, "task": task, "routed": routed,
            "log": list(log) if log is not None else None,
            "state": copy.deepcopy(state) if state is not None else None,
            "state_digest": _digest(state) if state is not None else None}


def run_state(T, seed):
    """Chapter 3 + SKILL.state: Algorithm 1 via the skill_state library."""
    shift, tag, captures = make_shift(T, seed), f"state|T={T}|s={seed}", []

    class ShiftEnv:
        def reset(self):
            self.i = 0
            return shift[0].text

        def step(self, action):
            self.i += 1
            done = self.i >= len(shift)
            return ("" if done else shift[self.i].text), done

    def on_step(s):                          # the trace: kept by the harness, never re-fed
        if _verb(s["action"]) == "WRITE":
            captures.append(capture("state", T, seed, s["t"], s["observation"], state=s["state"]))

    t0 = time.perf_counter()
    res = SkillStateRuntime(TRACKER_SKILL, OpenRouterLLM(TRACKER, f"track[{tag}]"), max_retries=2,
                            max_state_chars=6000, on_step=on_step).run(ShiftEnv(), max_steps=T)
    return {"harness": "state", "T": T, "seed": seed, "tag": tag, "captures": captures,
            "prompt_chars": [s.prompt_chars for s in res.steps],
            "rejected": res.rejected_patches, "parse_failures": 0,
            "final_state": res.final_state, "seconds": time.perf_counter() - t0}


REACT_TEMPLATE = """Instructions:
{instructions}

History:
{history}

Latest Observation: {observation}
Generate your next reasoning and action (format 'Action: <cmd>'):"""


def _fit(history, budget_chars):
    kept, used = [], 0
    for obs, resp in reversed(history):
        size = len(obs) + len(resp) + 40
        if used + size > budget_chars:
            break
        kept.append((obs, resp))
        used += size
    return kept[::-1]


def run_transcript(T, seed, budget_chars=None):
    """Chapter 3's harness over a stream: append everything (or keep a budget-matched window)."""
    harness = "transcript" if budget_chars is None else "window"
    shift, tag = make_shift(T, seed), f"{harness}|T={T}|s={seed}"
    history, captures, prompt_chars, parse_failures = [], [], [], 0
    t0 = time.perf_counter()
    for t, ev in enumerate(shift):
        kept = history
        if budget_chars is not None:
            fixed = len(REACT_TEMPLATE.format(instructions=P.text, history="", observation=ev.text))
            kept = _fit(history, max(0, budget_chars - fixed))
        hist = "\n".join(f"Observation: {o}\nReasoning & Action: {r}" for o, r in kept) or "(none yet)"
        prompt = REACT_TEMPLATE.format(instructions=P.text, history=hist, observation=ev.text)
        prompt_chars.append(len(prompt))
        text, _, _ = chat_call(model=TRACKER, prompt=prompt, label=f"track[{tag}]")
        found = re.findall(r"Action[\s*`:]*\b(WAIT|WRITE)\b", text, re.I)
        parse_failures += not found
        if (found[-1].upper() if found else "WAIT") == "WRITE":
            captures.append(capture(harness, T, seed, t, ev.text, log=[o for o, _ in kept] + [ev.text]))
        history.append((ev.text, text.strip()))
    return {"harness": harness, "T": T, "seed": seed, "tag": tag, "captures": captures,
            "prompt_chars": prompt_chars, "rejected": 0, "parse_failures": parse_failures,
            "final_state": None, "seconds": time.perf_counter() - t0}

## 5. One delayed correction, step by step

The state harness at one step. `SIGMA_BEFORE` is Σ as the stream has it
just before the perf team's correction arrives: four changes in the release, PR-415
still at "40 percent", the dark-mode revert already applied, the incident resolved.
It is written here as a fixture so the step costs exactly one call; its content is
what the events up to that point say.

Then the step itself, exactly as `SkillStateRuntime` does it: `build_prompt(P +
state spec, Σ, O_t)`, one tracker call, `parse_response`, `merge`, validate, commit.

**The point.** The correction is resolved on the step it arrives, by overwriting one
string. Nothing downstream ever sees the 40 again.

**Model produces** the patch; **harness computes** everything around it.

In [29]:
from skill_state import parse_response, ParseError

SIGMA_BEFORE = {
    "release": {"version": "2.4.0", "changes": {
        "PR-412": "the CSV exporter was rewritten; exports run about twice as fast",
        "PR-415": "peak memory during export reduced by roughly 40 percent",
        "PR-420": "the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}",
        "PR-421": "fixed escaping of quotes inside CSV fields"}},
    "incident": {"id": "INC-88", "status": "resolved", "timeline": {
        "14:02": "ingest API returned 503 for all regions", "14:20": "traffic shifted to the standby ingest cluster",
        "14:36": "ingest API recovered in eu-west", "15:07": "all regions recovered"}},
}
assert not state_errors(SIGMA_BEFORE)
O_T = next(ev.text for _, ev in CORE if "re-measured" in ev.text)          # the correction event, from the stream


def one_step(state, observation, label, max_retries=2):
    """One SKILL.state step on a given Σ: prompt, one tracker call, parse, merge, validate; retry with the error."""
    error = None
    for attempt in range(1, max_retries + 2):
        prompt = build_prompt(TRACKER_SKILL.instructions, state, observation, error)
        text, _, _ = chat_call(model=TRACKER, prompt=prompt, label=label)        # model produces
        try:
            parsed = parse_response(text)
            candidate = merge(state, parsed.state_patch)
            errors = state_errors(candidate)
            if errors:
                raise ParseError("; ".join(errors))
            return candidate, parsed, attempt, len(prompt)
        except (ParseError, TypeError) as exc:
            error = str(exc)                                              # Σ untouched; retry with the error only
    raise RuntimeError(f"no valid patch after {max_retries + 1} attempts: {error}")


SIGMA_AFTER, step, attempts, prompt_len = one_step(SIGMA_BEFORE, O_T, label="track[live|correction]")

print(f"O_t          : {O_T}")
print(f"prompt       : {ntok(P.text + STATE_SPEC)} tokens of instructions + Σ + O_t = {prompt_len} chars; attempts: {attempts}")
print(f"MODEL PRODUCED patch  : {json.dumps(step.state_patch)}")
print(f"MODEL PRODUCED action : {step.action}   (reasoning discarded: {len(step.reasoning)} chars)\n")
print("HARNESS: Σ before -> after, release.changes:")
before, after = SIGMA_BEFORE["release"]["changes"], SIGMA_AFTER["release"]["changes"]
for k in sorted(set(before) | set(after)):
    mark = "  " if before.get(k) == after.get(k) else "* "
    print(f"   {mark}{k}: {before.get(k, '(absent)')[:60]}")
    if before.get(k) != after.get(k):
        print(f"     -> {after.get(k, '(deleted)')[:60]}")
print(f"\nstate digest before {_digest(SIGMA_BEFORE)} -> after {_digest(SIGMA_AFTER)}; "
      f"'40 percent' in Σ after: {'40 percent' in json.dumps(SIGMA_AFTER)}")

O_t          : Perf team: PR-415 re-measured on production-size exports; the peak memory reduction is roughly 35 percent, not 40.
prompt       : 790 tokens of instructions + Σ + O_t = 4657 chars; attempts: 1
MODEL PRODUCED patch  : {"release": {"changes": {"PR-415": "peak memory during export reduced by roughly 35 percent"}}}
MODEL PRODUCED action : WAIT   (reasoning discarded: 192 chars)

HARNESS: Σ before -> after, release.changes:
     PR-412: the CSV exporter was rewritten; exports run about twice as f
   * PR-415: peak memory during export reduced by roughly 40 percent
     -> peak memory during export reduced by roughly 35 percent
     PR-420: the /v1/export endpoint now returns 202 with a job id instea
     PR-421: fixed escaping of quotes inside CSV fields

state digest before 12b0443e37d7 -> after 844b8a1768f6; '40 percent' in Σ after: False


## 6. Full comparison (replay; ~110 calls, ~90 s for the shifts)

The grid: the state harness first (its measured prompt size sets the
window's budget), then transcript and window. **Model produces** on every step (the
small tracker model); the captures and prompt sizes are **harness computes**.

With `RUN_FULL = False` this cell prints one line and keeps the saved outputs. To
reproduce the book's numbers, set `HORIZONS = (30, 90)`; the transcript harness then
makes about 120 calls per horizon.

In [30]:
HORIZONS = (30,)          # (30, 90) reproduces the book's numbers; 180 shows the gap widen
SEEDS = (0,)              # more seeds, more confidence, more cost
MAX_WORKERS = 4           # parallel shifts; lower it if OpenRouter rate-limits you
HARNESSES = ("transcript", "window", "state")


def run_of(harness, T, seed):
    return next((r for r in RUNS if r["harness"] == harness and r["T"] == T and r["seed"] == seed), None)


RUNS, BUDGET = [], {}

if RUN_FULL:
    def _safe(fn, *args):
        try:
            return fn(*args)
        except Exception as exc:                                  # noqa: BLE001
            return {"error": f"{type(exc).__name__}: {exc}"[:300], "args": args}


    grid = [(T, s) for T in HORIZONS for s in SEEDS]
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(MAX_WORKERS) as pool:              # state first: it sets the window's budget
        state_runs = list(pool.map(lambda a: _safe(run_state, *a), grid))
    BUDGET = {(r["T"], r["seed"]): int(sum(r["prompt_chars"]) / len(r["prompt_chars"]))
              for r in state_runs if "error" not in r}
    jobs = [(T, s, None) for T, s in grid] + [(T, s, BUDGET.get((T, s), 6000)) for T, s in grid]
    with cf.ThreadPoolExecutor(MAX_WORKERS) as pool:
        transcript_runs = list(pool.map(lambda j: _safe(run_transcript, *j), jobs))

    RUNS = [r for r in state_runs + transcript_runs if "error" not in r]
    for r in state_runs + transcript_runs:
        if "error" in r:
            print("FAILED RUN", r["args"], r["error"])
    RUNS.sort(key=lambda r: (r["T"], r["seed"], HARNESSES.index(r["harness"])))


    print(f"{len(RUNS)} shifts in {time.perf_counter() - t0:.0f}s; window budget (chars): {BUDGET}\n")
    print(f"{'T':>4} {'harness':<12}{'WRITE at':>12}{'due at':>12}{'rejected patches':>18}{'parse failures':>16}")
    for r in RUNS:
        due = [i for i, e in enumerate(make_shift(r["T"], r["seed"])) if e.kind == "write"]
        print(f"{r['T']:>4} {r['harness']:<12}{str([c['t'] for c in r['captures']]):>12}{str(due):>12}"
              f"{r['rejected']:>18}{r['parse_failures']:>16}")
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


### Benefit 1 (replay): cost that stops growing with the shift

Prompt tokens per step and cumulative over the shift. The transcript prompt grows
with every step; the state prompt holds P, Σ and one event.

In the full run at `T=90`: transcript 171,892 shift tokens, window 94,062, state
107,223; max prompt 3,118 vs 1,111 vs 1,122. At `T=30` the state harness is about as
expensive as the transcript, because its fixed instructions are larger. The gap is
arithmetic, not model quality.

In [31]:
if RUN_FULL:
    import matplotlib.pyplot as plt


    def track_rows(harness, T, seed):
        return [u for u in USAGE if u["label"] == f"track[{harness}|T={T}|s={seed}]"]


    T_MAX, S0 = max(HORIZONS), SEEDS[0]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for h in HARNESSES:
        per_call = [u["prompt_tokens"] for u in track_rows(h, T_MAX, S0)]
        if per_call:
            ax1.plot(per_call, label=h)
            ax2.plot([sum(per_call[:i + 1]) for i in range(len(per_call))], label=h)
    for i, e in enumerate(make_shift(T_MAX, S0)):
        if e.kind == "write":
            for ax in (ax1, ax2):
                ax.axvline(i, color="grey", ls=":", lw=1)
    ax1.set(title=f"Prompt tokens per step (T={T_MAX})", xlabel="step (patch retries included)", ylabel="tokens")
    ax2.set(title="Cumulative prompt tokens over the shift", xlabel="step", ylabel="tokens")
    ax1.legend(); ax2.legend(); plt.tight_layout(); plt.show()
    print("dotted lines: the two WRITE triggers\n")

    print(f"{'T':>4} {'harness':<12}{'first prompt':>13}{'last prompt':>13}{'max prompt':>12}{'shift tokens':>14}")
    for r in RUNS:
        rows = track_rows(r["harness"], r["T"], r["seed"])
        p = [u["prompt_tokens"] for u in rows]
        total = sum(u["prompt_tokens"] + u["completion_tokens"] for u in rows)
        print(f"{r['T']:>4} {r['harness']:<12}{p[0]:>13,}{p[-1]:>13,}{max(p):>12,}{total:>14,}")
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


## 7. What the writer is handed

### Definitions and the replay table

For each captured `WRITE`, count which traps are still present in the writer's input
and how many required facts are present. `X` marks a decision the writer would have
to make inside the same generation that writes the prose.

In the full run at `T=90`: transcript 13 of 14 traps in the writer input with all
8 facts; window 3 traps but only 2 facts (it forgot); state 0 traps, 8 facts. The
last column is a fact only the state harness can supply: the incident duration,
computed by the harness from Σ.

**Harness computes.** The definitions always run; the table is replay.

In [32]:
TRAP_MARKERS = {                       # must NOT reach a correct deliverable
    "dark mode":    r"dark mode",
    "40 %":         r"\b40 percent",
    "14:31":        r"14:31",
    "retries":      r"\bretries\b",
    "other branch": r"maintenance/|targets 2\.5\.0",
    "vendor":       r"\bvendor\b",
    "Globex":       r"globex",
}
RECORDED = ["14:02", "14:20", "14:36", "15:07"]
REQUIRED = {                           # MUST be available to write a correct deliverable
    "release-notes":   {"twice as fast": r"twice", "quote escaping": r"quot", "35 %": r"\b35 percent",
                        "job id": r"job id"},
    "incident-report": {t: re.escape(t) for t in RECORDED},
}


def _duration(timeline: Dict[str, str]) -> Optional[int]:
    keys = sorted(timeline)
    if len(keys) < 2:
        return None
    (h1, m1), (h2, m2) = (map(int, keys[0].split(":")), map(int, keys[-1].split(":")))
    return (h2 * 60 + m2) - (h1 * 60 + m1)


def writer_input(cap: Dict, skill: str) -> Tuple[Dict, str]:
    """(harness-owned facts for the surface, the user message): chapter 3 §5's tool result."""
    if cap["state"] is None:
        log = "\n".join(f"- {e}" for e in cap["log"])
        return dict(HARNESS_FACTS), f"Event log, oldest first:\n{log}\n\nTask: {cap['task']}"
    st = cap["state"]
    if skill == "release-notes":
        r = st.get("release", {})
        facts = {k: r[k] for k in ("version", "breaking") if r.get(k) is not None}
        body = "\n".join(f"- {c}" for _, c in sorted(r.get("changes", {}).items()))
        return facts, f"Release {r.get('version', '(version not recorded)')} of the Acme data platform.\n{body}"
    inc = st.get("incident", {})
    facts = {k: inc[k] for k in ("id", "status") if inc.get(k) is not None}
    if _duration(inc.get("timeline", {})) is not None:
        facts["duration_minutes"] = _duration(inc["timeline"])      # derived by the harness from Σ
    body = "\n".join(f"- {t} UTC — {e}" for t, e in sorted(inc.get("timeline", {}).items()))
    return facts, f"Incident {inc.get('id', '(id not recorded)')}. Recorded events, UTC, oldest first:\n{body}"


def exposure(cap, skill):
    facts, user = writer_input(cap, skill)
    text = (user + " " + json.dumps(facts)).lower()
    traps = {n: bool(re.search(rx, text)) for n, rx in TRAP_MARKERS.items()}
    have = sum(bool(re.search(rx, text)) for rx in REQUIRED[skill].values())
    return traps, have, ntok(user), facts

if RUN_FULL:
    EXPOSURE: Dict[Tuple[str, int, int, str], Dict] = {}
    for r in RUNS:
        for cap in r["captures"]:
            if cap["routed"] in WRITER_AXES:
                traps, have, tok, facts = exposure(cap, cap["routed"])
                EXPOSURE[(r["harness"], r["T"], r["seed"], cap["routed"])] = {
                    "traps": traps, "have": have, "tok": tok, "derived": "duration_minutes" in facts}

    for T in HORIZONS:
        print(f"\nT={T}, seed={SEEDS[0]}   (X = trap present in the writer's input)")
        print(f"   {'deliverable':<17}{'harness':<12}" + "".join(f"{n:>13}" for n in TRAP_MARKERS)
              + f"{'facts':>8}{'input tok':>11}  derived")
        for skill in ("incident-report", "release-notes"):
            for h in HARNESSES:
                e = EXPOSURE.get((h, T, SEEDS[0], skill))
                if e is None:
                    print(f"   {skill:<17}{h:<12}  (no capture: the harness did not WRITE, or misrouted)")
                    continue
                print(f"   {skill:<17}{h:<12}" + "".join(f"{'X' if e['traps'][n] else '.':>13}" for n in TRAP_MARKERS)
                      + f"{str(e['have']) + '/' + str(len(REQUIRED[skill])):>8}{e['tok']:>11}  "
                      + ("duration" if e["derived"] else "-"))
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


In [33]:
if RUN_FULL:
    # The same moment, side by side: what the chapter-3 writer is handed, and what it is handed with Σ.
    T_MAX, S0 = max(HORIZONS), SEEDS[0]
    for h in ("transcript", "state"):
        r = run_of(h, T_MAX, S0)
        due = next(i for i, e in enumerate(make_shift(T_MAX, S0)) if e.expect == "release-notes")   # the moment the notes were due
        cap = next((c for c in (r["captures"] if r else []) if c["t"] == due and c["routed"] == "release-notes"), None)
        if cap is None:
            continue
        facts, user = writer_input(cap, "release-notes")
        lines = user.splitlines()
        print(f"--- release-notes input, {h} harness, T={T_MAX}: {len(lines)} lines, {ntok(user)} tokens, facts={facts}")
        shown = lines if len(lines) <= 14 else lines[:6] + [f"   ... {len(lines) - 12} more lines ..."] + lines[-6:]
        print(textwrap.indent("\n".join(shown), "   "), "\n")
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


### The writer input from the live Σ

`capture()` is the harness's own function from section 4: it routes the
task (one small router call) and stores Σ. `writer_input()` turns Σ into the facts
and the user message the writer will get. `exposure()` counts traps and required
facts in it, before any writer runs.

**Model produces** the routing choice; **harness computes** the input and the counts.

In [34]:
SHIP = next(ev for _, ev in CORE if ev.expect == "release-notes")                  # the release trigger, from the stream
LIVE_CAP = capture("live", 0, 0, len(CORE) - 1, SHIP.text, state=SIGMA_AFTER)     # one router call, Σ stored
assert LIVE_CAP["routed"] == "release-notes", LIVE_CAP["routed"]

traps, have, tok, facts = exposure(LIVE_CAP, "release-notes")
facts_, user_ = writer_input(LIVE_CAP, "release-notes")
print(f"routed to {LIVE_CAP['routed']}; state digest {LIVE_CAP['state_digest']}\n")
print("what the writer will be handed (user message):")
print(textwrap.indent(user_, "   "))
print(f"\nharness facts: {facts_}")
print(f"traps present: {[n for n, v in traps.items() if v] or 'none'}   required facts present: {have}/{len(REQUIRED['release-notes'])}   {tok} tokens")

routed to release-notes; state digest 844b8a1768f6

what the writer will be handed (user message):
   Release 2.4.0 of the Acme data platform.
   - the CSV exporter was rewritten; exports run about twice as fast
   - peak memory during export reduced by roughly 35 percent
   - the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}
   - fixed escaping of quotes inside CSV fields

harness facts: {'version': '2.4.0'}
traps present: none   required facts present: 4/4   78 tokens


## 8. The deliverables

### Definitions and the replay grading

Chapter 3's writing path, unchanged: route, `load_for_task`, `compile_surface` with
the task's axes, one typed call to the writer at temperature 0. Only the input
differs. The graders are plain functions; each names a trap or a rule.

**Model produces** the deliverables; **harness computes** the grades. Definitions
always run; the six deliverables and their trap table are replay.

In [35]:
class IncidentReport(BaseModel):
    summary: str = Field(description="Two sentences.")
    timeline: List[str] = Field(description="One line per recorded event, each starting with its UTC time, oldest first.")


WHOLE_SCHEMA = {"release-notes": ReleaseNotes, "incident-report": IncidentReport}


def write(cap: Dict, skill: str, label: str):
    """Chapter 3's writer: compile the surface for this task, one typed call."""
    facts, user = writer_input(cap, skill)
    surface_ = compile_surface(artifacts=load_for_task([skill, "plainspoken"]),
                               axes=WRITER_AXES[skill], facts=facts)
    out = typed_call(model=WRITER, response_model=WHOLE_SCHEMA[skill], temperature=0.0, label=label,
                     messages=[{"role": "system", "content": surface_.text},
                               {"role": "user", "content": user}])
    return out, surface_.manifest["surface_digest"], user + " " + json.dumps(facts)


def grade_release(n: ReleaseNotes) -> Dict[str, bool]:
    text = n.summary + " " + " ".join(n.highlights)
    low = text.lower()
    return {
        "covers exporter speed":            bool(re.search(r"twice|two times|\b2x\b|2×", low)),
        "covers quote escaping":            "quot" in low,
        "covers export memory":             "memory" in low,
        "covers export retrieval (job)":    "job" in low,
        "current number (35, not 40)":      bool(re.search(r"\b35\b", low)) and not re.search(r"(?<!not )\b40\b", low),
        "no reverted change (dark mode)":   "dark mode" not in low,
        "no moved change (retries)":        not re.search(r"\bretr(y|ies)\b", low),
        "no other-branch change":           not re.search(r"time ?zone|cron|gzip|last-used", low),
        "no named customer":                "globex" not in low,
        "customer: no status code/path":    not re.search(r"\b202\b|/v1/", text),
        "contract: 3-6 bullets, <=25 words": 3 <= len(n.highlights) <= 6 and len(n.summary.split()) <= 25,
        "no '!' or superlative":            "!" not in text and not re.search(r"\b(best|fastest|most powerful)\b", low),
    }


def grade_incident(r: IncidentReport) -> Dict[str, bool]:
    times = [m.group(1) for line in r.timeline if (m := re.match(r"\s*(\d{2}:\d{2})", line))]
    text = r.summary + " " + " ".join(r.timeline)
    low = text.lower()
    return {
        "all recorded times present":       all(t in times for t in RECORDED),
        "superseded 14:31 absent":          "14:31" not in text,
        "only recorded events":             set(times) <= set(RECORDED),
        "every line starts with a time":    bool(r.timeline) and len(times) == len(r.timeline),
        "oldest first":                     times == sorted(times),
        "duration stated (65 minutes)":     bool(re.search(r"\b65[ -]minute", low)),
        "no cause speculation / vendor":    not re.search(r"vendor|load balancer", low),
        "no named customer":                "globex" not in low,
        "no apology opener":                not re.match(r"\s*(we('| a)re sorry|we apologi|sorry|apolog)", r.summary.lower()),
        "summary: two sentences":           len(re.findall(r"[.!?](?=\s|$)", r.summary.strip())) == 2,
    }


GRADERS = {"release-notes": grade_release, "incident-report": grade_incident}


def deliver(run: Dict, skill: str) -> Dict:
    due = next(i for i, e in enumerate(make_shift(run["T"], run["seed"])) if e.expect == skill)
    cap = next((c for c in run["captures"] if c["t"] == due), None)
    row = {"harness": run["harness"], "T": run["T"], "seed": run["seed"], "skill": skill,
           "output": None, "checks": None, "score": 0.0, "note": None, "input": None,
           "surface": None, "state_digest": None}
    if cap is None:
        row["note"] = f"no WRITE at t={due}"
    elif cap["routed"] != skill:
        row["note"] = f"routed to {cap['routed']}"
    else:
        try:
            out, digest, used = write(cap, skill, f"write[{run['tag']}|{skill}]")
            checks = GRADERS[skill](out)
            row.update(output=out, checks=checks, score=sum(checks.values()) / len(checks), input=used,
                       surface=digest, state_digest=cap["state_digest"])
        except Exception as exc:                              # noqa: BLE001
            row["note"] = f"{type(exc).__name__}: {exc}"[:200]
    return row

DELIVERED = []
if RUN_FULL:
    with cf.ThreadPoolExecutor(MAX_WORKERS) as pool:
        DELIVERED = list(pool.map(lambda j: deliver(*j), [(r, skill) for r in RUNS for skill in GRADERS]))

    for d in DELIVERED:
        if d["note"]:
            print(f"   {d['harness']:<11} T={d['T']:<4} {d['skill']:<17} scored 0: {d['note']}")
    print(f"{len(DELIVERED)} deliverables written and graded")
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


In [36]:
if RUN_FULL:
    # Which trap each harness fell into. '.' pass, 'X' fail, '-' not written.
    for skill in ("incident-report", "release-notes"):
        rows = [d for d in DELIVERED if d["skill"] == skill and d["seed"] == SEEDS[0]]
        names = next((list(d["checks"]) for d in rows if d["checks"]), [])
        cols = [(h, T) for T in HORIZONS for h in HARNESSES]
        print(f"\n{skill}")
        print(f"   {'check':<36}" + "".join(f"{h[:5] + '@' + str(T):>11}" for h, T in cols))
        for name in names:
            cells = []
            for h, T in cols:
                d = next((d for d in rows if d["harness"] == h and d["T"] == T), None)
                cells.append("-" if d is None or d["checks"] is None else ("." if d["checks"][name] else "X"))
            print(f"   {name:<36}" + "".join(f"{c:>11}" for c in cells))
        print(f"   {'score':<36}" + "".join(
            f"{next((d['score'] for d in rows if d['harness'] == h and d['T'] == T), 0):>11.2f}" for h, T in cols))

    T_MAX, S0 = max(HORIZONS), SEEDS[0]
    for h in ("transcript", "state"):
        d = next((d for d in DELIVERED if d["harness"] == h and d["T"] == T_MAX and d["seed"] == S0
                  and d["skill"] == "release-notes" and d["output"]), None)
        if d:
            print(f"\n--- release notes, {h} harness, T={T_MAX}")
            print(f"  summary  : {d['output'].summary}")
            for line in d["output"].highlights:
                print(f"  highlight: {line}")
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


### One writer call from the live Σ

The same `write()` as above, on the live capture. The grader runs on it too, so
you can compare it with the replay table.

**Model produces** the notes; **harness computes** the grades.

In [37]:
LIVE_NOTES, LIVE_SURFACE_DIGEST, LIVE_INPUT = write(LIVE_CAP, "release-notes", label="write[live|release-notes]")
LIVE_D = {"harness": "live", "T": 0, "seed": 0, "skill": "release-notes", "output": LIVE_NOTES,
          "checks": grade_release(LIVE_NOTES), "input": LIVE_INPUT, "surface": LIVE_SURFACE_DIGEST,
          "state_digest": LIVE_CAP["state_digest"], "note": None}
LIVE_D["score"] = sum(LIVE_D["checks"].values()) / len(LIVE_D["checks"])

print("MODEL PRODUCED")
print(f"   summary  : {LIVE_NOTES.summary}")
for h in LIVE_NOTES.highlights:
    print(f"   highlight: {h}")
print(f"\nHARNESS: surface {LIVE_SURFACE_DIGEST}, state {LIVE_D['state_digest']}, score {LIVE_D['score']:.2f}")
for name, ok in LIVE_D["checks"].items():
    print(f"   [{'.' if ok else 'X'}] {name}")

MODEL PRODUCED
   summary  : We rebuilt the CSV exporter, so exports run about twice as fast with lower peak memory use.
   highlight: Export about twice as fast
   highlight: Use roughly 35 percent less peak memory during export
   highlight: Poll `/v1/export/{id}` after `/v1/export` returns a job id
   highlight: Escape quotes correctly inside CSV fields

HARNESS: surface 0a0fd88ffaed, state 844b8a1768f6, score 0.92
   [.] covers exporter speed
   [.] covers quote escaping
   [.] covers export memory
   [.] covers export retrieval (job)
   [.] current number (35, not 40)
   [.] no reverted change (dark mode)
   [.] no moved change (retries)
   [.] no other-branch change
   [.] no named customer
   [X] customer: no status code/path
   [.] contract: 3-6 bullets, <=25 words
   [.] no '!' or superlative


## 9. A test the harness can now run

**Provenance**: every number and time in a deliverable must appear in the writer's
input. Against a transcript this is nearly toothless, because the stale `40` and
`14:31` are *in* the log. Against Σ it catches exactly the superseded values. The
cell plants the stale value into each real deliverable (a mutation test) and asks
whether that harness's provenance test catches it.

In the full run: planted stale values caught 0 of 2 for the transcript harness,
2 of 2 for the state harness.

**Harness computes.** Definitions always run; the table over the six deliverables is replay.

In [38]:
NUM = r"\d+(?:[.:]\d+)*"
STALE = {"release-notes": r"(?<!not )\b40\b", "incident-report": r"14:31"}


def provenance(output, input_text) -> List[str]:
    text = " ".join(v if isinstance(v, str) else " ".join(v) for v in output.model_dump().values())
    return sorted(set(re.findall(NUM, text)) - set(re.findall(NUM, input_text)))


def plant_stale(d):
    """The mutation: the superseded value, written into an otherwise real deliverable."""
    o = d["output"]
    if d["skill"] == "release-notes":
        return ReleaseNotes(summary=o.summary, highlights=o.highlights + ["Use about 40 percent less memory on big exports"])
    return IncidentReport(summary=o.summary, timeline=o.timeline + ["14:31 UTC — ingest API recovered in eu-west"])

MUTATION = {}
if RUN_FULL:
    MUTATION: Dict[Tuple[str, int, int, str], bool] = {}
    print(f"{'T':>4} {'harness':<12}{'deliverable':<17}{'real output':>16}{'planted stale value':>22}   trace")
    for d in DELIVERED:
        if d["output"] is None:
            continue
        real = provenance(d["output"], d["input"])
        caught = bool({"40", "14:31"} & set(provenance(plant_stale(d), d["input"])))
        MUTATION[(d["harness"], d["T"], d["seed"], d["skill"])] = caught
        trace = f"surface={d['surface']}" + (f" state={d['state_digest']}" if d["state_digest"] else " state=—")
        print(f"{d['T']:>4} {d['harness']:<12}{d['skill']:<17}{('flags ' + str(real)) if real else 'passes':>16}"
              f"{'CAUGHT' if caught else 'missed':>22}   {trace}")

    print("\n'missed' means the stale value is in that harness's input, so the test cannot tell it\n"
          "from a real fact. That is the transcript harness's normal case, by construction.")

    st = run_of("state", max(HORIZONS), SEEDS[0])
    if st:
        print(f"\nthe facts behind every state-harness deliverable at T={max(HORIZONS)} "
              f"({len(json.dumps(st['final_state']))} chars, final Σ):")
        print(textwrap.indent(json.dumps(st["final_state"], indent=2), "   "))
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


In [39]:
# everyone runs: the same test on the live deliverable
print(f"live release notes, provenance on the real output : {provenance(LIVE_NOTES, LIVE_INPUT) or 'passes'}")
print(f"live release notes, planted stale 40              : "
      f"{'CAUGHT' if '40' in provenance(plant_stale(LIVE_D), LIVE_INPUT) else 'missed'}   (the 40 is not in Σ, so the test can see it)")

live release notes, provenance on the real output : passes
live release notes, planted stale 40              : CAUGHT   (the 40 is not in Σ, so the test can see it)


## 10. The scorecard (replay)

The four benefits per horizon on one page. Earlier rows depend least on the models,
later rows most. One seed is a demonstration, not a statistic.

In [40]:
if RUN_FULL:
    def scorecard(T):
        card = {}
        for h in HARNESSES:
            rows = [u for s in SEEDS for u in USAGE if u["label"] == f"track[{h}|T={T}|s={s}]"]
            exp = [e for (hh, TT, s, _), e in EXPOSURE.items() if hh == h and TT == T]
            dl = [d for d in DELIVERED if d["harness"] == h and d["T"] == T]
            n = len(SEEDS)
            card[h] = {
                "shift tokens (tracker)":        sum(u["prompt_tokens"] + u["completion_tokens"] for u in rows) / n,
                "max prompt tokens":             max((u["prompt_tokens"] for u in rows), default=0),
                "traps in writer input (of 14)": sum(sum(e["traps"].values()) for e in exp) / n,
                "required facts present (of 8)": sum(e["have"] for e in exp) / n,
                "writer input tokens":           sum(e["tok"] for e in exp) / n,
                "deliverable score (mean)":      sum(d["score"] for d in dl) / max(len(dl), 1),
                "stale values in deliverables":  sum(bool(d["output"] and re.search(STALE[d["skill"]],
                                                     " ".join(str(v) for v in d["output"].model_dump().values())))
                                                     for d in dl) / n,
                "planted stale values caught":   sum(v for (hh, TT, s, _), v in MUTATION.items() if hh == h and TT == T) / n,
                "fact digest in trace":          "yes" if h == "state" else "no",
            }
        return card


    for T in HORIZONS:
        card = scorecard(T)
        print(f"\nT={T}")
        print(f"   {'':34}" + "".join(f"{h:>16}" for h in HARNESSES))
        for metric in card["state"]:
            vals = [card[h][metric] for h in HARNESSES]
            fmt = [f"{v:>16,.0f}" if isinstance(v, (int, float)) and v >= 100 else
                   f"{v:>16.2f}" if isinstance(v, float) else f"{str(v)[:15]:>16}" for v in vals]
            print(f"   {metric:<34}" + "".join(fmt))

    # The same, as a picture across horizons.
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for ax, metric in zip(axes, ["shift tokens (tracker)", "traps in writer input (of 14)", "deliverable score (mean)"]):
        for i, h in enumerate(HARNESSES):
            ax.bar([x + i * 0.27 for x in range(len(HORIZONS))], [scorecard(T)[h][metric] for T in HORIZONS],
                   width=0.27, label=h)
        ax.set_xticks([x + 0.27 for x in range(len(HORIZONS))], [f"T={T}" for T in HORIZONS])
        ax.set_title(metric)
    axes[0].legend(); plt.tight_layout(); plt.show()
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


**Reading the T=30 result honestly.** In the saved run, at 30 events the transcript
harness scored 1.00 on the graders, ahead of explicit state at
0.96, and used slightly fewer tracker tokens (33,249 vs
35,033). A strong writer handed a 30-line log resolves the traps itself.

And yet the state harness handed its writer all 8 measured facts with 0 planted
distractions (the transcript handed 13 of 14 traps), and its provenance check caught
both planted stale values where the transcript's caught none. That is the lesson at
this horizon: clean state makes the writer's job easier and makes certain checks
possible at all, even when a short transcript run produces better prose on the
rubric. The cost argument only appears at longer horizons (T=90 above).

## 11. A judge that names one failed part

The graders above know the seven traps because we planted them. A
production harness does not. So a **judge** reads the deliverable against the
*compiled rules it was written under* (the same `compile_surface` output, section by
section, each with a rule id) and the input it was given, and returns a typed list of
findings: `rule_id`, `part`, `quote`, `explanation`. The judge is a different model
family from the writer (`JUDGE`), at temperature 0, through the same `typed_call`.

The deliverable is the live release notes from section 8. The judge runs twice: on
the **real** output, and on the **planted** version (the stale `40 percent` bullet
written into the real output by `plant_stale`). The mutation is a harness-made test
input, not a faked model answer; the judge's verdict on it is real, and because we
planted the fault we can check the verdict against ground truth: the quoted words
must contain the `40`. That checked finding is the case for section 12. If the real
output also carries findings, they are printed for comparison.

One implementation note. The judge call uses `instructor.Mode.JSON` instead of the
writer's tool-calling mode. With a forced tool call, the judge models tested tended
to return a "finding" on a compliant deliverable and withdraw it inside the
explanation, so the list was never empty. In JSON mode an empty list is a normal
answer.

**Harness computes** the rule list and the ground-truth check; **model produces** the verdict.

In [41]:
from typing import Literal


class Finding(BaseModel):
    rule_id: str = Field(description="The id of the violated rule, copied exactly from between the square brackets "
                                     "in RULES, including the part before the colon.")
    part: Literal["summary", "highlights"] = Field(description="The part of the deliverable the quoted words are in.")
    quote: str = Field(description="The exact words from the deliverable that violate the rule.")
    explanation: str = Field(description="One sentence: which requirement of the rule the quoted words break.")


class Verdict(BaseModel):
    findings: List[Finding] = Field(description="Only certain violations. Empty if the deliverable follows every rule.")


def surface_for(cap: Dict, skill: str) -> Surface:
    """Exactly the surface write() compiled for this capture (same inputs, same digest)."""
    facts, _ = writer_input(cap, skill)
    return compile_surface(artifacts=load_for_task([skill, "plainspoken"]), axes=WRITER_AXES[skill], facts=facts)


def rules_in(surface: Surface) -> Dict[str, str]:
    """rule_id -> the rule text as compiled for this task (sliced to its axes)."""
    out = {}
    for s in surface.manifest["sections"]:
        a = ARTIFACTS[s["artifact"]]
        body = a.sections[s["section"]]
        out[f"{a.name}:{s['section']}"] = slice_section(body, surface.manifest["axes"]) if a.axes else body
    return out


JUDGE_SYSTEM = """You are a verifier for one deliverable. Check it against RULES exactly as written.

A finding is a violation you are certain of: the quoted words break a requirement that a rule states, and
the deliverable would have to change because of it. If you are not certain, or a rule does not apply to the
part the words are in, report nothing for it. Never report something and then describe it as acceptable.

For each finding give: the rule id copied exactly from between the square brackets in RULES (including the
part before the colon), the part of the deliverable the quoted words are in ('summary' or 'highlights'),
the offending words quoted verbatim, and one sentence naming the requirement they break.

INPUT is the only source of facts. A number, date, version or endpoint in the deliverable that is not in
INPUT violates the identity's refusals. If nothing is violated, return an empty list."""


def judge(output: BaseModel, surface: Surface, input_text: str, label: str) -> Verdict:
    rule_text = "\n\n".join(f"[{rid}]\n{txt}" for rid, txt in rules_in(surface).items())
    return typed_call(model=JUDGE, response_model=Verdict, temperature=0.0, label=label,
                      mode=instructor.Mode.JSON,           # see the note above: an empty list must be a normal answer
                      messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                {"role": "user", "content": f"RULES\n{rule_text}\n\nINPUT\n{input_text}\n\n"
                                                            f"DELIVERABLE\n{output.model_dump_json(indent=2)}"}])


D, CAP = LIVE_D, LIVE_CAP
SURF = surface_for(CAP, D["skill"])
assert SURF.manifest["surface_digest"] == D["surface"], "recompiled surface differs from the one the writer saw"
print(f"deliverable: release-notes, live Σ, surface {D['surface']}, {len(rules_in(SURF))} rules\n")

VERDICTS = {}
for name, out in (("real", D["output"]), ("planted", plant_stale(D))):
    v = judge(out, SURF, D["input"], label=f"judge[live|{name}]")
    VERDICTS[name] = (out, v)
    print(f"--- {name} deliverable: {len(v.findings)} finding(s)")
    for f in v.findings:
        print(f"   [{f.rule_id}] part={f.part}\n      quote: \"{f.quote}\"\n      {f.explanation}")
    print()

# ground truth: we planted the 40, so the judge's finding on the planted version can itself be checked
PLANTED_VALUE = "40"
hit = next((f for f in VERDICTS["planted"][1].findings if PLANTED_VALUE in f.quote), None)
print(f"ground-truth check on the planted verdict: {'the judge quoted the planted value' if hit else 'the judge MISSED the planted value'}")

if hit is not None:
    CASE, FINDING = "planted", hit
elif VERDICTS["real"][1].findings:
    CASE, FINDING = "real", VERDICTS["real"][1].findings[0]
else:
    CASE, FINDING = None, None

if CASE is None:
    print("no finding to work with; section 12 will report that and make no repair")
else:
    ORIGINAL, BEFORE = VERDICTS[CASE]
    print(f"case for the focused retry: the {CASE} deliverable; flagged part: {FINDING.part}; cited rule: {FINDING.rule_id}")

deliverable: release-notes, live Σ, surface 0a0fd88ffaed, 14 rules

--- real deliverable: 0 finding(s)

--- planted deliverable: 2 finding(s)
   [acme-docs-agent:identity.refusals] part=highlights
      quote: "Use about 40 percent less memory on big exports"
      The refusals forbid stating any number not in the input, and the input gives only the roughly 35 percent memory reduction, not 40 percent.
   [release-notes:highlights.structure] part=highlights
      quote: "Use about 40 percent less memory on big exports"
      Every highlight must trace to a supplied change, and no supplied change reports a 40 percent memory reduction, so this bullet is an invented change.

ground-truth check on the planted verdict: the judge quoted the planted value
case for the focused retry: the planted deliverable; flagged part: highlights; cited rule: acme-docs-agent:identity.refusals


## 12. Extract the part, retry it, gate it

The harness does not hand the whole deliverable back to the writer. It:

1. takes the **flagged part** only (`summary` or `highlights`);
2. compiles a **repair surface**: the rules that govern that part (a fixed, written
   down mapping) plus the cited rule, each as compiled for this task, plus the same
   harness facts;
3. calls the writer with a response model that has **only that field**;
4. runs four **gates**, all in code except the fourth, which is the same judge again:
   1. the cited rule exists in the compiled surface;
   2. only the flagged part came back;
   3. every number or date in the repaired deliverable is in the input (`provenance`);
   4. a re-check clears the finding without adding one: the set of `(rule_id, part)`
      findings after the repair no longer contains the targeted pair, and contains no
      pair that was absent before.

On any failed gate the original is kept.

**Harness computes** the surface, gates 1 to 3, the set comparison in gate 4, the
splice and the decision; **model produces** the repaired part and the re-check.

In [42]:
@lru_cache(maxsize=1)
def pricing() -> Dict[str, Tuple[float, float]]:
    req = urllib.request.Request(f"{BASE_URL}/models",
                                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
    data = json.load(urllib.request.urlopen(req, timeout=60))["data"]
    return {m["id"]: (float(m["pricing"]["prompt"]), float(m["pricing"]["completion"])) for m in data}


def cost_of(u) -> float:
    p_in, p_out = pricing().get(u["model"], (0.0, 0.0))
    return u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out


class SummaryOnly(BaseModel):
    summary: str = Field(description="One sentence, at most 25 words.")


class HighlightsOnly(BaseModel):
    highlights: List[str] = Field(description="Three to six bullets, one change each.")


PART_MODEL = {"summary": SummaryOnly, "highlights": HighlightsOnly}

GOVERNS = {   # which compiled rules govern each part; a harness decision, written down once
    "summary":    ["acme-docs-agent:identity.refusals", "release-notes:summary.style", "release-notes:address.contract",
                   "release-notes:prohibitions.hard", "release-notes:typography", "plainspoken:persona.voice"],
    "highlights": ["acme-docs-agent:identity.refusals", "release-notes:highlights.structure", "release-notes:address.contract",
                   "release-notes:prohibitions.hard", "release-notes:typography", "plainspoken:persona.voice"],
}


def repair_surface(part: str, finding: Finding, surface: Surface, facts: Dict) -> str:
    rules = rules_in(surface)
    ids = [r for r in GOVERNS[part] if r in rules] + ([finding.rule_id] if finding.rule_id in rules
                                                       and finding.rule_id not in GOVERNS[part] else [])
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items())
    return ("\n\n".join(f"[{rid}]\n{rules[rid]}" for rid in ids)
            + f"\n\n## Facts supplied by the harness\n\n{fact_lines}\n\nThese are given. Do not restate them as "
              "claims of your own, and do not infer any fact that is not listed.")


def focused_retry(original: BaseModel, finding: Finding, cap: Dict, skill: str, surface: Surface, label: str):
    facts, user = writer_input(cap, skill)
    system = repair_surface(finding.part, finding, surface, facts)
    task = (f"Rewrite only the `{finding.part}` of these release notes. A verifier found this violation:\n"
            f"  rule [{finding.rule_id}]: {finding.explanation}\n  offending words: \"{finding.quote}\"\n"
            f"Fix that and change nothing else. Return only `{finding.part}`.\n\n"
            f"CURRENT {finding.part.upper()}\n{json.dumps(getattr(original, finding.part), indent=2)}\n\nINPUT\n{user}")
    patch = typed_call(model=WRITER, response_model=PART_MODEL[finding.part], temperature=0.0, label=label,
                       messages=[{"role": "system", "content": system}, {"role": "user", "content": task}])
    return patch, system


def pairs(verdict: Verdict):
    return {(f.rule_id, f.part) for f in verdict.findings}


def gates(original, patch, finding, surface, input_text, before: Verdict, label: str):
    results, candidate = {}, original.model_copy(update=patch.model_dump())
    results["1 cited rule exists in the compiled surface"] = finding.rule_id in rules_in(surface)
    results["2 only the flagged part came back"] = set(patch.model_dump()) == {finding.part} and all(
        getattr(candidate, k) == getattr(original, k) for k in original.model_dump() if k != finding.part)
    results["3 numbers/dates in the repaired deliverable are in the input"] = not provenance(candidate, input_text)
    after = judge(candidate, surface, input_text, label)                      # model produces
    target = (finding.rule_id, finding.part)
    results["4 re-check clears the finding without adding one"] = (target not in pairs(after)) and (pairs(after) <= pairs(before))
    return results, candidate, after


if CASE is None:
    print("no finding to repair; skipping the focused retry")
else:
    tag = f"live|{CASE}"
    n0 = len(USAGE)
    PATCH, REPAIR_SYSTEM = focused_retry(ORIGINAL, FINDING, CAP, D["skill"], SURF, label=f"repair[{tag}]")
    RESULTS, CANDIDATE, AFTER = gates(ORIGINAL, PATCH, FINDING, SURF, D["input"], BEFORE, label=f"recheck[{tag}]")
    REPAIR_USAGE = USAGE[n0:]
    ACCEPTED = all(RESULTS.values())
    FINAL = CANDIDATE if ACCEPTED else ORIGINAL

    print(f"repair surface: {ntok(REPAIR_SYSTEM)} tokens (full surface was {ntok(SURF.text)}); "
          f"part returned by the writer: {list(PATCH.model_dump())}\n")
    print("MODEL PRODUCED (repaired part)")
    print(textwrap.indent(json.dumps(PATCH.model_dump(), indent=2), "   "))
    print("\nGATES")
    for name, ok in RESULTS.items():
        print(f"   [{'pass' if ok else 'FAIL'}] {name}")
    print(f"   gate 4 detail: target {(FINDING.rule_id, FINDING.part)}")
    print(f"      findings before: {sorted(pairs(BEFORE))}")
    print(f"      findings after : {sorted(pairs(AFTER)) or '(none)'}")
    print(f"      new findings   : {sorted(pairs(AFTER) - pairs(BEFORE)) or '(none)'}")
    print(f"\ndecision: {'repair accepted' if ACCEPTED else 'repair refused; original kept'}")
    print(f"\nrepair loop tokens (writer + judge re-check): {sum(u['prompt_tokens'] for u in REPAIR_USAGE):,} prompt, "
          f"{sum(u['completion_tokens'] for u in REPAIR_USAGE):,} completion, USD {sum(cost_of(u) for u in REPAIR_USAGE):.5f}")

repair surface: 445 tokens (full surface was 783); part returned by the writer: ['highlights']

MODEL PRODUCED (repaired part)
   {
     "highlights": [
       "Export about twice as fast",
       "Use roughly 35 percent less peak memory during export",
       "Poll `/v1/export/{id}` after `/v1/export` returns a job id",
       "Escape quotes correctly inside CSV fields",
       "Use roughly 35 percent less memory on big exports"
     ]
   }

GATES
   [pass] 1 cited rule exists in the compiled surface
   [pass] 2 only the flagged part came back
   [pass] 3 numbers/dates in the repaired deliverable are in the input
   [pass] 4 re-check clears the finding without adding one
   gate 4 detail: target ('acme-docs-agent:identity.refusals', 'highlights')
      findings before: [('acme-docs-agent:identity.refusals', 'highlights'), ('release-notes:highlights.structure', 'highlights')]
      findings after : (none)
      new findings   : (none)

decision: repair accepted

repair loop tokens (wri

### Full regeneration for comparison (replay)

The whole deliverable rewritten from the same capture, then judged, so scope, tokens
and cost can be set side by side with the focused retry.

In [43]:
if RUN_FULL:
    if CASE is not None:
        n1 = len(USAGE)
        REGEN, _, _ = write(CAP, D["skill"], label=f"regen[{tag}]")           # model produces
        regen_verdict = judge(REGEN, SURF, D["input"], label=f"rejudge[{tag}]")
        REGEN_USAGE = USAGE[n1:]

        def tot(rows, key): return sum(u[key] for u in rows)
        print(f"{'':<28}{'focused retry':>18}{'full regeneration':>28}")
        print(f"{'parts touched':<28}{str([FINDING.part]):>18}{str(list(REGEN.model_dump())):>28}")
        print(f"{'surface handed to writer':<28}{ntok(REPAIR_SYSTEM):>18,}{ntok(SURF.text):>28,}")
        print(f"{'model calls (writer+judge)':<28}{len(REPAIR_USAGE):>18}{len(REGEN_USAGE):>28}")
        print(f"{'prompt tokens':<28}{tot(REPAIR_USAGE, 'prompt_tokens'):>18,}{tot(REGEN_USAGE, 'prompt_tokens'):>28,}")
        print(f"{'completion tokens':<28}{tot(REPAIR_USAGE, 'completion_tokens'):>18,}{tot(REGEN_USAGE, 'completion_tokens'):>28,}")
        print(f"{'USD':<28}{sum(cost_of(u) for u in REPAIR_USAGE):>18.5f}{sum(cost_of(u) for u in REGEN_USAGE):>28.5f}")
        print(f"{'judge findings after':<28}{len(AFTER.findings):>18}{len(regen_verdict.findings):>28}")
        if regen_verdict.findings:
            print("   regeneration findings: " + "; ".join(f"[{f.rule_id}] {f.part}: {f.quote[:50]}" for f in regen_verdict.findings))
else:
    print('replay: the outputs below were saved from an execution with RUN_FULL = True')

replay: the outputs below were saved from an execution with RUN_FULL = True


**Two different things the focused retry buys, and only one of them is tokens.**

- **Limiting what can change.** The writer was handed a repair surface of
  445 tokens instead of the full 783, with a response model
  that has one field. The `summary` could not be touched, and the gates checked that
  it was not. That is the real win: a smaller blast radius, and a reviewable one.
- **Saving tokens across the whole repair loop.** Writer plus judge re-check cost
  5,767 prompt tokens against 5,918 for regeneration plus its
  judge call: a difference of about 150 tokens. In this run the retry even cost more in
  dollars ($0.00227 vs $0.00178), because the judge's re-check produced a long
  answer. The judge reads the whole deliverable and the whole rule set either way.
  Do not sell the focused retry as a cost measure.

What the gates cannot do: gate 3 only catches numbers and dates, gate 4 is itself a
model, and the `GOVERNS` mapping is a harness author's decision. All three are
written down, which is what makes them reviewable.

## 13. What this pass cost

Every tracker, router, writer, judge and repair call is in `USAGE`. In a session with
`RUN_FULL = False`, only the `live` rows are yours.

In [44]:
agg: Dict[Tuple[str, str], Dict[str, float]] = {}
for u in USAGE:
    role, _, rest = u["label"].partition("[")
    a = agg.setdefault((role, rest.split("|")[0]), {"n": 0, "in": 0, "out": 0, "usd": 0.0})
    a["n"] += 1; a["in"] += u["prompt_tokens"]; a["out"] += u["completion_tokens"]; a["usd"] += cost_of(u)

print(f"{'role':<8}{'harness':<14}{'calls':>7}{'tok in':>12}{'tok out':>10}{'USD':>10}")
for (role, who), a in sorted(agg.items()):
    print(f"{role:<8}{who:<14}{a['n']:>7}{a['in']:>12,}{a['out']:>10,}{a['usd']:>10.5f}")
print(f"{'total':<22}{sum(a['n'] for a in agg.values()):>7}{sum(a['in'] for a in agg.values()):>12,}"
      f"{sum(a['out'] for a in agg.values()):>10,}{sum(a['usd'] for a in agg.values()):>10.5f}")
print("\nPriced from the published per-token rates against measured tokens; treat it as the\n"
      "cost of the work, not a reconciliation of an account.")

role    harness         calls      tok in   tok out       USD
judge   live                2      10,169     3,487   0.00276
recheck live                1       4,977     2,520   0.00176
repair  live                1         760       157   0.00034
route   live                1         204        37   0.00032
track   live                1       1,098       104   0.00129
write   live                1         947       142   0.00036
total                       7      18,155     6,447   0.00683

Priced from the published per-token rates against measured tokens; treat it as the
cost of the work, not a reconciliation of an account.


## 14. What we built, in order

| harness part | what it decides | without a model |
|---|---|---|
| the state schema (Σ) as an artifact | what facts a task can hold, and what gets deleted | yes |
| validation of every patch | whether an update can corrupt what we know | yes |
| Σ handed to the writer, not the log | what the writer is told is true | yes |
| the provenance test | whether the output invented or kept a stale fact | yes |
| the judge | which rule, which part, which words | no: a second model |
| the focused retry and its gates | how much of the deliverable is re-opened, and whether the repair is accepted | gates 1 to 3 yes; gate 4 no |

**Takeaways.**

- Carry state, not history. The transcript harness hands every trap to the writer by
  construction; Σ hands none, and keeps the facts (section 7).
- A correction is resolved on the step it arrives, by one patch (section 5).
- A provenance test only works once the input holds current facts (section 9).
- When something is wrong, name the part. Repair that part under its own rules, and
  let code decide whether to accept the repair (section 12).
- The trace holds the surface digest (which rules), the state digest (which facts)
  and now the judge's finding and the gate results.

To run the full comparison, set `RUN_FULL = True` at the top (and `HORIZONS = (30, 90)`
in section 6 for the book's numbers) and re-run; expect several minutes.